## Agent Use Case with DSPy

- Setting up a query answering assistant using DSPy
- We will define generic answer tool and maths tool
- The module in DSPy will work as follows:
    - Given a query first decide the style to answer in
    - Based on type of query, call the appropriate tool to get the answer
    - Agent formats the answer and returns

## Configuring the Language Model to use

In [3]:

# Set up language model 
# Using a gemma2 model through ollama
import dspy
lm = dspy.LM("ollama_chat/gemma2:9b", api_base="http://localhost:11434", api_key="")
dspy.configure(lm=lm)
dspy.configure_cache(enable_memory_cache=False, enable_disk_cache=False)

## Defining the Signatures to be used

In [4]:
# Signature to take query as input and identify the type of the query and provide guidance on how it is supposed to be answered
class QueryType(dspy.Signature):
    """Classify the question to choose answer formatting."""
    question: str = dspy.InputField(desc="The question asked by the user. Based on the question, determine the query type and guidance for answering.")
    query_type: str = dspy.OutputField(desc="Based on the question, determine the query type. The query type should be one among the following: Factual One line Answer, Comparison Table, Analytical Bullet Points, Mathematical Question")
    guidance: str = dspy.OutputField(desc="Any formatting instructions for the answer based on the query type.")

In [5]:
# Signature to take query as input and answer the question based on the query type and formatting instructions
class AnswerWithStyle(dspy.Signature):
    question: str = dspy.InputField(desc="The question asked by the user.")
    answer: str = dspy.OutputField(desc="The answer to the question and the answer format should be aligned with the query type and formatting instructions.")

## Defining the Tools

In [6]:
# Math tool to evaluate simple math expressions
# Takes as input a string with the mathematical expression and returns the result as a string
import math
import re

def math_tool(expression: str) -> str:
    """
    Evaluate a simple math expression safely.
    Supports +, -, *, /, **, parentheses, and common math functions.

    The input to the tool should be a string with the mathemaical experession such as just "8 * 4" or "12 * (7 + 5)"
    """
    # very small safe eval using allowed names
    allowed = {
        "pi": math.pi,
        "e": math.e,
        "sqrt": math.sqrt,
        "log": math.log,
        "log10": math.log10,
        "sin": math.sin,
        "cos": math.cos,
        "tan": math.tan,
        "abs": abs,
        "round": round,
    }
    # basic sanity check
    if not re.fullmatch(r"[0-9+\-*/()., eplogsinctarbsqr]+", expression.replace("**", "")):
        return "Error: unsupported characters in expression"
    try:
        return str(eval(expression, {"__builtins__": {}}, allowed))
    except Exception as e:
        return f"Error: {e}"

# General answer tool to answer non mathematical questions

def general_answer_tool(question) -> str:
    """
    For answering of non mathematical questions, use this tool.
    Provide the question as a detailed string with all details.
    """
    try:
        # Use the configured LM to answer the question
        lm = dspy.settings.lm
        prompt = f"Answer the following question concisely and accurately:\n\nQuestion: {question}\n\nAnswer:"
        response = lm(prompt)
        
        # Extract the text from the response
        if hasattr(response, 'choices') and len(response.choices) > 0:
            return response.choices[0].message.content.strip()
        elif isinstance(response, str):
            return response.strip()
        else:
            return str(response).strip()
    except Exception as e:
        return f"Error: Unable to answer - {e}"

## Defining the DSPy Module

The module defined will make use of the predictors. The first one will be a .Predict predictor followed by a .ReACT predictor. We will bring these 2 predictors together using a dspy.Module base class

In [9]:
class QueryAssistant(dspy.Module):
    def __init__(self):
        super().__init__()
        self.style_decider = dspy.Predict(QueryType) 
        self.react = dspy.ReAct(AnswerWithStyle, tools=[math_tool, general_answer_tool])

    def forward(self, question: str):
        decision = self.style_decider(question=question)
        print("decision", decision)

        complete_question = f"Question Type: {decision.query_type}\nFormatting Instructions: {decision.guidance}\nQuestion: {question}"
        
        return self.react(question=complete_question)

# Initializing the assistant

In [10]:
query_assistant = QueryAssistant()

# Executing and seeing how the module performs

In [11]:
answer = query_assistant(question="What is the capital of France?")

decision Prediction(
    query_type='Factual One line Answer',
    guidance='Provide a concise, one-line answer.'
)


In [12]:
answer

Prediction(
    trajectory={'thought_0': "France is a country in Europe, so its capital is likely a major city. I'll ask a general knowledge tool to provide the answer.", 'tool_name_0': 'general_answer_tool', 'tool_args_0': {'question': 'What is the capital of France?'}, 'observation_0': "['Paris  \\n']", 'thought_1': "Paris is the capital of France according to the observation. I'm done.", 'tool_name_1': 'finish', 'tool_args_1': {}, 'observation_1': 'Completed.'},
    reasoning='I used a general knowledge tool to find the capital of France.',
    answer='Paris'
)

In [13]:
answer_2 = query_assistant(question="What is 1448*76*5?")
answer_2

decision Prediction(
    query_type='Mathematical Question',
    guidance='Provide the numerical answer only.'
)


Prediction(
    trajectory={'thought_0': 'This is a multiplication problem. I can use the math tool to calculate the answer.', 'tool_name_0': 'math_tool', 'tool_args_0': {'expression': '1448*76*5'}, 'observation_0': '550240', 'thought_1': 'The answer is 550240.', 'tool_name_1': 'finish', 'tool_args_1': {}, 'observation_1': 'Completed.'},
    reasoning='This is a multiplication problem. I used a math tool to calculate the product of 1448, 76, and 5.',
    answer='550240'
)

In [11]:
answer_3 = query_assistant(question="Compare Oceans and Seas")

decision Prediction(
    query_type='Comparison Table',
    guidance='Provide a comparison table outlining the key differences between oceans and seas. Consider aspects like size, depth, salinity, location, and wave activity.'
)


In [12]:
print(answer_3)

Prediction(
    trajectory={'thought_0': 'To compare oceans and seas effectively, I need to gather information about their size, depth, salinity, location, and wave activity. A comparison table format would be suitable for presenting this information.', 'tool_name_0': 'general_answer_tool', 'tool_args_0': {'question': 'Provide information about the key differences between oceans and seas. Consider aspects like size, depth, salinity, location, and wave activity.'}, 'observation_0': '["Oceans are vastly larger and deeper than seas, encompassing over 70% of Earth\'s surface. They have higher average salinities due to increased evaporation and less freshwater input.  Seas are partially enclosed by land, often connected to oceans but experiencing more restricted water flow and variable salinity levels. Wave activity is generally stronger in open oceans due to wider fetch distances, while seas can experience calmer waters influenced by coastal geography. \\n"]', 'thought_1': 'I have gathered

In [19]:
print(answer_3.answer)

| Feature        | Ocean                                     | Sea                             |
|----------------|---------------------------------------------|---------------------------------|
| **Size**       | Vast, covering over 70% of Earth's surface | Relatively smaller             |
| **Depth**      | Extremely deep, average depth ~3,688 meters | Generally shallower than oceans   |
| **Salinity**  | Higher average salinity (due to evaporation) | Variable salinity levels         |
| **Location**   | Open bodies of water, often interconnected   | Partially enclosed by land     |
| **Wave Activity**| Stronger wave activity due to wider fetch  | Calmer waters influenced by coastline |


In [ ]:
# Inspecting the messages passed to the language model
query_assistant.inspect_history(n=1)





[2026-02-23T21:25:01.007737]

System message:

Your input fields are:
1. `question` (str): The question asked by the user.
2. `trajectory` (str):
Your output fields are:
1. `reasoning` (str): 
2. `answer` (str): The answer to the question and the answer format should be aligned with the query type and formatting instructions.
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## question ## ]]
{question}

[[ ## trajectory ## ]]
{trajectory}

[[ ## reasoning ## ]]
{reasoning}

[[ ## answer ## ]]
{answer}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Given the fields `question`, produce the fields `answer`.


User message:

[[ ## question ## ]]
Question Type: Comparison Table
Formatting Instructions: Provide a comparison table outlining the key differences between oceans and seas. Consider aspects like size, depth, salinity, location, and wave activity.
Question: Compare Oceans and Seas

[[ ## traje

In [ ]:
answer_4 = query_assistant(question="Explain in depth how an internal combustion engine works.")

In [40]:
print(answer_4.answer)

## How an Internal Combustion Engine Works: 

**1. Intake Stroke:**
* The piston moves downwards, creating a vacuum in the cylinder.
* The intake valve opens, allowing a mixture of air and fuel to be drawn into the cylinder.

**2. Compression Stroke:**
* The intake valve closes.
* The piston moves upwards, compressing the air-fuel mixture, increasing its temperature and pressure significantly. 

**3. Power (Combustion) Stroke:**
* At the peak of compression, a spark plug ignites the compressed air-fuel mixture.
* The resulting explosion pushes the piston downwards with great force. This is the power stroke that drives the crankshaft.

**4. Exhaust Stroke:**
* The exhaust valve opens.
* The piston moves upwards, pushing the burnt gases (exhaust) out of the cylinder and into the exhaust system. 

**5. Cycle Repeats:**
* The cycle then repeats itself with a new intake of air-fuel mixture, starting again with the intake stroke. 



**Additional Notes:**

* **Crankshaft:** The reciprocating

In [38]:
print(answer_4)

Prediction(
    trajectory={'thought_0': "I need to explain how an internal combustion engine works.  That involves several steps: intake, compression, combustion, and exhaust. I'll break it down into bullet points as requested.", 'tool_name_0': 'general_answer_tool', 'tool_args_0': {'question': 'Explain in depth how an internal combustion engine works. Provide a comprehensive explanation in bullet point format. Each bullet should cover a distinct aspect or stage of the process.'}, 'observation_0': "['##  How an Internal Combustion Engine Works: \\n\\n**1. Intake Stroke:**\\n* The piston moves downwards, creating a vacuum in the cylinder.\\n* The intake valve opens, allowing a mixture of air and fuel to be drawn into the cylinder.\\n\\n**2. Compression Stroke:**\\n* The intake valve closes.\\n* The piston moves upwards, compressing the air-fuel mixture, increasing its temperature and pressure significantly. \\n\\n**3. Power (Combustion) Stroke:**\\n* At the peak of compression, a spark